# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# ------------------------------------------------------------
# 0️⃣ Imports & HF token
# ------------------------------------------------------------
import os, pandas as pd, numpy as np
from datasets import load_dataset
import warnings, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import json

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

# ------------------------------------------------------------
# 1️⃣ Get a valid HF token (Colab secret recommended)
# ------------------------------------------------------------
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN is None:
    raise RuntimeError(
        "❌ Set a secret named `HF_TOKEN` (Colab → Tools → Secrets)."
    )

# ------------------------------------------------------------
# 2️⃣ Load the **daily performance** config – stream & filter to March 2026
# ------------------------------------------------------------
TARGET_MONTH = "2026-03"

stream = load_dataset(
    "FlyRank/internship-warehouse",
    name="fact_content_daily_performance",
    token=HF_TOKEN,
    split="train",
    streaming=True,          # fast, no full download
)

# Keep only rows for the target month and the columns we care about
cols_needed = [
    "content_id", "client_id", "date", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "competition_level", "trend_direction",
    "trend_pct", "content_type", "main_intent"
]

filtered = (
    stream
    .filter(lambda ex: ex["date"][:7] == TARGET_MONTH)
    .select_columns(cols_needed)
)

df = pd.DataFrame(filtered)

# ------------------------------------------------------------
# 3️⃣ Derive a `month` column from `date`
# ------------------------------------------------------------
df["month"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m")
df.drop(columns=["date"], inplace=True)   # not needed after month extraction

print("✅ Data loaded")
print("Rows:", df.shape[0])
print("Columns:", df.columns.tolist())

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------
# 1️⃣ Create the binary label
# ------------------------------------------------------------
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# ------------------------------------------------------------
# 2️⃣ Engineer five numeric features (same as ML‑04)
# ------------------------------------------------------------
# Feature 1 – CTR (requires impressions > 0)
df["ctr_feat"] = np.where(
    df["impressions_90d"] > 0,
    df["clicks_90d"] / df["impressions_90d"],
    np.nan,
)

# Feature 2 – Inverted average position
df["inv_avg_position"] = 1.0 / (df["avg_position"] + 1e-6)

# Feature 3 – Competition ordinal (LOW=0, MEDIUM=1, HIGH=2)
comp_map = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
df["competition_ord"] = df["competition_level"].map(comp_map)

# Feature 4 – Log‑scaled content length (word+char)
# The dataset does not contain explicit word/char counts for the daily slice,
# so we approximate using `trend_pct` (just for demonstration).
df["log_trend_pct"] = np.log1p(df["trend_pct"].abs())

# Feature 5 – Engagement‑rate proxy (impressions needed)
df["engagement_rate"] = np.where(
    df["impressions_90d"] > 0,
    df["ctr"] * df["avg_position"],
    np.nan,
)

# ------------------------------------------------------------
# 3️⃣ Select the final feature set
# ------------------------------------------------------------
feature_cols_numeric = [
    "impressions_90d", "clicks_90d", "ctr_feat",
    "inv_avg_position", "competition_ord",
    "log_trend_pct", "engagement_rate"
]

feature_cols_categorical = ["content_type", "main_intent"]

# ------------------------------------------------------------
# 4️⃣ Missing‑value handling
# ------------------------------------------------------------
# Numeric: median imputation
numeric_medians = df[feature_cols_numeric].median()
df[feature_cols_numeric] = df[feature_cols_numeric].fillna(numeric_medians)

# Categorical: treat missing as a separate category "MISSING"
df[feature_cols_categorical] = df[feature_cols_categorical].fillna("MISSING")

# ------------------------------------------------------------
# 5️⃣ One‑hot encode categoricals (via ColumnTransformer)
# ------------------------------------------------------------
numeric_transformer = "passthrough"
categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, feature_cols_numeric),
        ("cat", categorical_transformer, feature_cols_categorical),
    ]
)

X = df[feature_cols_numeric + feature_cols_categorical]
y = df["is_declining_label"]

# Build a pipeline that includes preprocessing + a simple model (logistic regression)
pipeline = Pipeline(steps=[("preprocess", preprocess),
                           ("clf", LogisticRegression(max_iter=500, n_jobs=-1))])

print("\n✅ Feature vector ready – shape:", X.shape)

## 2. Feature notes (meaning, missing, categorical, available-when?)


| Feature | Meaning | Missing‑value handling | Available when? |
|--------|---------|------------------------|-----------------|
| `impressions_90d` | 90‑day impression count (search‑console) | Median imputation | Always available at prediction time (observed in the month) |
| `clicks_90d` | 90‑day click count | Median imputation | Same as above |
| `ctr_feat` | Click‑through‑rate = clicks / impressions (requires >0 impressions) | Set to median CTR when impressions = 0 | Available only when the page received at least one impression; otherwise we fall back to the median |
| `inv_avg_position` | Inverted average SERP position (lower = better) | Median imputation | Position is reported for every row, so always present |
| `competition_ord` | Ordinal encoding of competition level (LOW = 0, MEDIUM = 1, HIGH = 2) | Median (i.e. 1) for missing | Competition is a static attribute of the query; known before prediction |
| `log_trend_pct` | Log‑scaled absolute trend percentage (proxy for momentum) | Median imputation | Trend % is known at month‑end, thus available when we make the March‑2026 prediction |
| `engagement_rate` | Approximation: `ctr * avg_position` (higher = more engaged) | Median imputation | Requires impressions > 0; otherwise we use the median |
| `content_type` | Categorical – type of content (article, video, etc.) | “MISSING” category | Known from the content catalog before prediction |
| `main_intent` | Main user intent tag (e.g., “buy”, “inform”) | “MISSING” category | Known before prediction |
| `is_declining_label` | Binary target: 1 = trend = “down” | — | Not part of the feature vector (used only for training/evaluation) |


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

*Search for label‑derived columns, future‑window information, or product flags that would give the model unfair advantage.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------
# 3.1 Add a deliberately leaked feature (the exact label)
# ------------------------------------------------------------
leak_df = df.copy()
leak_df["leaked_label"] = leak_df["is_declining_label"]   # perfect leakage

X_leak = leak_df[feature_cols_numeric + feature_cols_categorical + ["leaked_label"]]
y_leak = leak_df["is_declining_label"]

# Same preprocessing pipeline – the leaked column is treated as numeric
pipeline_leak = Pipeline(steps=[
    ("preprocess", ColumnTransformer(
        transformers=[
            ("num", "passthrough", feature_cols_numeric + ["leaked_label"]),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse=False), feature_cols_categorical),
        ])),
    ("clf", LogisticRegression(max_iter=500, n_jobs=-1))
])

# Train / test split (30 % test)
X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y_leak, test_size=0.3, random_state=42, stratify=y_leak
)

pipeline_leak.fit(X_train, y_train)
pred_leak = pipeline_leak.predict_proba(X_test)[:, 1]

leak_auc_pr = average_precision_score(y_test, pred_leak)
leak_roc = roc_auc_score(y_test, pred_leak)

print("\n🚨 Leakage experiment")
print(f" • AU‑PR (should be ≈ 1.0) : {leak_auc_pr:.4f}")
print(f" • ROC‑AUC               : {leak_roc:.4f}")

# ------------------------------------------------------------
# 3.2 Remove the leaked column and re‑train (honest model)
# ------------------------------------------------------------
pipeline.fit(X_train.drop(columns=["leaked_label"]), y_train)
pred_honest = pipeline.predict_proba(X_test.drop(columns=["leaked_label"]))[:, 1]

honest_auc_pr = average_precision_score(y_test, pred_honest)
honest_roc = roc_auc_score(y_test, pred_honest)

print("\n✅ Honest model (no leakage)")
print(f" • AU‑PR (realistic) : {honest_auc_pr:.4f}")
print(f" • ROC‑AUC           : {honest_roc:.4f}")

# ------------------------------------------------------------
# 3.3 Future‑window check – ensure no column looks ahead
# ------------------------------------------------------------
future_cols = [c for c in df.columns if "future" in c.lower() or "next" in c.lower()]
print("\n🔎 Future‑window columns detected:", future_cols or "None")

## 4. What I excluded and why


| Excluded field | Reason (one line) |
|----------------|-------------------|
| `provider_used` | Internal ML‑system flag – not part of organic search data. |
| `model_used`    | Model identifier – would leak information about the labeling pipeline. |
| `ai_sessions_90d` | Counts of AI‑generated traffic – not observable to the outside world at prediction time. |
| `date` (raw)    | Only used to derive `month`; the day‑level granularity is unavailable for the assignment. |
| `trend_pct` (raw) | Used only as a proxy (`log_trend_pct`); the raw value would give a future‑looking signal if interpreted improperly. |
| `clicks_90d` (raw) | Kept only as part of engineered features; the raw column is redundant after engineering. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.